# Практическое занятие 5. Классификация частичных разрядов по PRPD-признакам

Цель занятия - научиться строить диагностическую классификацию
состояния изоляции по признакам частичных разрядов и сравнить
несколько базовых классификаторов.

Частичный разряд - локальный электрический разряд, который
возникает в части изоляционной системы и не полностью перекрывает
промежуток между электродами. PRPD (Phase Resolved Partial
Discharge) - фазово-разрешенное представление частичных разрядов,
связывающее импульсы с фазой питающего напряжения.

## Инициализация среды выполнения

Ячейка ниже обеспечивает запуск блокнота в Google Colab и в
локальном Jupyter Notebook. Если проект уже открыт локально,
повторное клонирование не выполняется.

In [ ]:
# COLAB_BOOTSTRAP_APPailab
from pathlib import Path
import os
import sys

REQUIRED_PROCESSED_FILES = ['practice_05_partial_discharge_features.csv', 'practice_05_partial_discharge_diagnostics.csv']
PROJECT_REPOSITORY_URL = "https://github.com/Alexflex/appailab.git"

def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-colab.txt").exists() and (candidate / "data" / "processed").exists():
            return candidate
    return None

project_root = find_project_root(Path.cwd())

if project_root is None:
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB:
        workdir = Path("/content/appailab")
        if not workdir.exists():
            !git clone -q {PROJECT_REPOSITORY_URL} {workdir}
        project_root = workdir
        os.chdir(project_root)
        !pip install -q -r requirements-colab.txt
    else:
        raise FileNotFoundError(
            "Не найден корень проекта. Откройте блокнот из репозитория appailab "
            "или выполните git clone перед запуском."
        )

sys.path.insert(0, str(project_root / "src"))
missing_files = [
    name for name in REQUIRED_PROCESSED_FILES
    if not (project_root / "data" / "processed" / name).exists()
]
if missing_files:
    raise FileNotFoundError(
        "Не найдены подготовленные CSV: " + ", ".join(missing_files)
        + ". Выполните python scripts/generate_datasets.py."
    )

print(f"Корень проекта: {project_root}")
print("Проверенные CSV:", ", ".join(REQUIRED_PROCESSED_FILES))

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

DATA_DIR = project_root / "data" / "processed"
RANDOM_STATE = 20260507

## Теоретический блок

Метод опорных векторов (Support Vector Machine, SVM) строит
разделяющую поверхность между классами. Для линейного SVM эта
поверхность является гиперплоскостью. Для SVM с радиальной
базисной функцией (Radial Basis Function, RBF) граница может быть
нелинейной.

Логистическая регрессия (Logistic Regression) служит простой
базовой моделью классификации. Случайный лес и дерево решений
позволяют получить более гибкую нелинейную границу, но требуют
контроля переобучения.

Для многоклассовой диагностики важны не только accuracy, но и
macro-precision, macro-recall и macro-F1. Приставка macro означает,
что метрика сначала считается по каждому классу, а затем
усредняется без учета размера класса.

In [ ]:
FEATURES_FILE = DATA_DIR / "practice_05_partial_discharge_features.csv"
DIAGNOSTICS_FILE = DATA_DIR / "practice_05_partial_discharge_diagnostics.csv"

df = pd.read_csv(FEATURES_FILE)
diagnostics_df = pd.read_csv(DIAGNOSTICS_FILE)
full_df = df.merge(diagnostics_df, on="sample_id", validate="one_to_one")

display(df.head())
print("Размер feature-таблицы:", df.shape)
print("Классы:", sorted(df["defect_class"].unique()))

## Структура данных

В feature-CSV находится целевая переменная `defect_class`.
Остальные столбцы описывают статистические признаки PRPD:
фазовое положение, разброс фазы, число импульсов, кажущийся заряд,
повторяемость и параметры формы импульса.

В diagnostics-CSV находятся производные диагностические величины:
числовой код класса, бинарный признак наличия разряда и риск-оценка.
Эти столбцы не должны использоваться как входные признаки базовой
модели.

In [ ]:
display(df.describe(include="all").T)
class_counts = df["defect_class"].value_counts().sort_index()
display(class_counts.to_frame("count"))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(class_counts.index, class_counts.values, color="#4c78a8")
axes[0].set_title("Распределение классов")
axes[0].set_xlabel("Класс")
axes[0].set_ylabel("Число наблюдений")

for label, group in df.groupby("defect_class"):
    axes[1].scatter(group["phase_mean_deg"], group["mean_charge_pc"], s=18, alpha=0.65, label=label)
axes[1].set_title("PRPD-пространство: фаза и средний заряд")
axes[1].set_xlabel("Средняя фаза, deg")
axes[1].set_ylabel("Средний кажущийся заряд, pC")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
corr_features = [
    "phase_std_deg",
    "pulse_count",
    "mean_charge_pc",
    "max_charge_pc",
    "repetition_rate_hz",
    "prpd_entropy",
]
pd.plotting.scatter_matrix(df[corr_features], figsize=(12, 12), diagonal="hist", alpha=0.35)
plt.suptitle("Парные зависимости основных PRPD-признаков", y=1.02)
plt.show()

## Выбор признаков и подготовка выборок

In [ ]:
# TODO: заполните список признаков. не включайте defect_code, risk_score, high_risk и has_partial_discharge
# Рекомендуемые признаки: ['voltage_kv', 'frequency_hz', 'phase_mean_deg', 'phase_std_deg', 'pulse_count', 'mean_charge_pc', 'max_charge_pc', 'charge_iqr_pc', 'repetition_rate_hz', 'positive_negative_ratio', 'waveform_rise_ns', 'waveform_width_ns', 'prpd_entropy']
pd_features = None
if pd_features is None:
    raise ValueError('Заполните pd_features: не включайте defect_code, risk_score, high_risk и has_partial_discharge')

target_column = "defect_class"
forbidden_columns = {"defect_class", "defect_code", "risk_score", "high_risk", "has_partial_discharge"}
leaked = forbidden_columns.intersection(pd_features)
if leaked:
    raise ValueError(f"Обнаружена утечка данных: {sorted(leaked)}")

X = df[pd_features]
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)
print("Train:", X_train.shape, "Test:", X_test.shape)
display(y_test.value_counts().sort_index().to_frame("test_count"))

## Обучение классификаторов

In [ ]:
classifiers = {
    "logistic_regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "svm_linear": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="linear", C=1.0, class_weight="balanced")),
        ]
    ),
    "svm_rbf": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=3.0, gamma="scale", class_weight="balanced")),
        ]
    ),
}

predictions = {}
metric_rows = []
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    metric_rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision_macro": precision_score(y_test, pred, average="macro"),
            "recall_macro": recall_score(y_test, pred, average="macro"),
            "f1_macro": f1_score(y_test, pred, average="macro"),
        }
    )

metrics_df = pd.DataFrame(metric_rows).sort_values("f1_macro", ascending=False)
display(metrics_df)

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_pred = predictions[best_model_name]
labels = sorted(y.unique())

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    labels=labels,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
)
ax.set_title(f"Матрица ошибок: {best_model_name}")
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_pred, digits=3))

In [ ]:
rf_model = classifiers["random_forest"]
importance_df = pd.DataFrame(
    {
        "feature": pd_features,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df["feature"], importance_df["importance"], color="#f28e2b")
ax.set_title("Важность признаков случайного леса")
ax.set_xlabel("Относительная важность")
plt.tight_layout()
plt.show()
display(importance_df.sort_values("importance", ascending=False))

## Демонстрация утечки данных

> **Внимание. АНТИПРИМЕР - НЕ ИСПОЛЬЗОВАТЬ КАК РАБОЧУЮ МОДЕЛЬ.**
> В следующей ячейке в признаки добавляется `risk_score`.
> Этот столбец рассчитан из скрытого правила генератора и
> частично кодирует диагностический класс. Его использование
> завышает качество модели и нарушает учебную постановку.

In [ ]:
leakage_df = df.merge(diagnostics_df[["sample_id", "risk_score"]], on="sample_id", validate="one_to_one")
X_leak = leakage_df[pd_features + ["risk_score"]]
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)
leakage_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
leakage_model.fit(X_train_leak, y_train_leak)
leakage_pred = leakage_model.predict(X_test_leak)
print("F1-macro строгой лучшей модели:", round(float(metrics_df.iloc[0]["f1_macro"]), 4))
print("F1-macro модели с утечкой:", round(f1_score(y_test_leak, leakage_pred, average="macro"), 4))

## Самостоятельный эксперимент

In [ ]:
# TODO: задайте значение параметра. рекомендуемый диапазон C для SVM: 0.1..10.0
# Рекомендуемое значение для первого запуска: 3.0
experiment_c = None
if experiment_c is None:
    raise ValueError('Заполните experiment_c: рекомендуемый диапазон C для SVM: 0.1..10.0')

experiment_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=experiment_c, gamma="scale", class_weight="balanced")),
    ]
)
experiment_model.fit(X_train, y_train)
experiment_pred = experiment_model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, experiment_pred), 4))
print("Recall macro:", round(recall_score(y_test, experiment_pred, average="macro"), 4))
print("F1 macro:", round(f1_score(y_test, experiment_pred, average="macro"), 4))

## Задание для отчета

1. Опишите физический смысл PRPD-представления.
2. Сравните логистическую регрессию, дерево решений, случайный лес
   и два варианта SVM.
3. Объясните, почему масштабирование признаков обязательно для SVM.
4. Проанализируйте матрицу ошибок: какие классы чаще путаются.
5. Укажите, почему `risk_score` является антипримером утечки.

Открытые источники для расширения: Mendeley Data `Partial
Discharge Signals in Insulated Power Cables with Time-of-Arrival
Annotations` и наборы PRPD-изображений частичных разрядов. Для
реальных сигналов необходимо отдельно описывать частоту
дискретизации, способ фильтрации и протокол аннотирования.